# Data Preprocessing for Machine Learning

In [2]:
import pandas as pd
import numpy as np

df_cleaned = pd.read_csv("../../data/processed/berlin_airbnb_cleaned.csv")

print(df_cleaned.shape)
df_cleaned.head()

(4999, 46)


,index,Review ID,review_date,Reviewer ID,Reviewer Name,Comments,Listing ID,Listing URL,Listing Name,Host ID,...,Last Review,Overall Rating,Accuracy Rating,Cleanliness Rating,Checkin Rating,Communication Rating,Location Rating,Value Rating,Instant Bookable,Business Travel Ready
0,425975,148759129.0,05-01-17,6424668.0,Jean-Charles,Anna's place is as good as described.\nVery cl...,8247088,https://www.airbnb.com/rooms/8247088,ROOM IN WELL CONNECTED APARTMENT IN CENTER,20594022,...,01-05-19,94.0,10.0,9.0,10.0,10.0,10.0,9.0,f,f
1,391704,275999337.0,06-12-18,56097579.0,Ramy,"TrÃ¨s bel appartement, bien situÃ©, recommandÃ©!",6695238,https://www.airbnb.com/rooms/6695238,SchÃ¶ne 1-Zimmer-Wohnung in F-Hain,19044451,...,09-22-18,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f,f
2,317860,449159387.0,05-06-19,46216027.0,Laura Lind,Alina and her boyfriend are both very friendly...,33473307,https://www.airbnb.com/rooms/33473307,Private Room at Volkspark Prenzlauer Berg,78887435,...,05-12-19,100.0,10.0,10.0,10.0,10.0,10.0,10.0,t,f
3,240086,295857076.0,07-24-18,35058155.0,Oscar,"Great location, so arranging for check in and ...",24181070,https://www.airbnb.com/rooms/24181070,Lovely room in the heart of Friedrichshain,53305731,...,05-10-19,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f,f
4,292949,423863258.0,03-15-19,63790591.0,Isabel,Sehr schÃ¶ne Unterkunft mit toller zentraler L...,29527972,https://www.airbnb.com/rooms/29527972,Apartment 1 in Kreuzberg at GÃ¶rlitzer Park,190957759,...,05-13-19,99.0,10.0,10.0,10.0,10.0,10.0,9.0,t,f


In [3]:
## Step 1 - Remove Unnecessary Columns
columns_to_drop = [
    # Index
    'index',

    # Review information
    'Review ID',
    'review_date',
    'Reviewer ID',
    'Reviewer Name',
    'Comments',

    # IDs
    'Listing ID',
    'Host ID',

    # URLs
    'Listing URL',
    'Host URL',

    # Names
    'Listing Name',
    'Host Name',

    # Dates
    'Host Since',

    # Location not useful
    'City',
    'Postal Code'
]

df_ml = df_cleaned.drop(columns=columns_to_drop, errors='ignore')

print(df_ml.shape)

(4999, 31)


In [4]:
## Step 2 - Explore the Target Variable (Price)
print(df_ml['Price'].describe())

print("\nHighest prices:")
print(df_ml['Price'].nlargest(10))

print("\nZero prices:")
print((df_ml['Price'] <= 0).sum())

count    4999.000000
mean       68.352270
std        54.042753
min        10.000000
25%        37.000000
50%        53.000000
75%        80.000000
max       888.000000
Name: Price, dtype: float64

Highest prices:
1196    888.0
3707    580.0
2613    522.0
919     506.0
4980    506.0
547     500.0
571     500.0
3257    500.0
4303    500.0
4902    460.0
Name: Price, dtype: float64

Zero prices:
0


In [5]:
MAX_PRICE = 300

df_ml = df_ml[
    (df_ml['Price'] > 0) &
    (df_ml['Price'] <= MAX_PRICE)
]

print(df_ml.shape)

(4970, 31)


In [6]:
df_ml.drop(
    columns=['First Review', 'Last Review'],
    errors='ignore',
    inplace=True
)

In [7]:
# Text columns
df_ml['Host Response Time'] = df_ml['Host Response Time'].fillna('Unknown')
df_ml['Is Superhost'] = df_ml['Is Superhost'].fillna('False')

In [8]:
rating_columns = [
    'Host Response Rate',
    'Accuracy Rating',
    'Cleanliness Rating',
    'Checkin Rating',
    'Communication Rating',
    'Location Rating',
    'Value Rating'
]

for col in rating_columns:

    # Remove the % sign
    df_ml[col] = (
        df_ml[col]
        .astype(str)
        .str.replace('%', '', regex=False)
    )

    # Convert to numeric
    df_ml[col] = pd.to_numeric(df_ml[col], errors='coerce')

    # Fill missing values with the median
    df_ml[col] = df_ml[col].fillna(df_ml[col].median())

In [9]:
print(df_ml.isnull().sum())

Host Response Time       0
Host Response Rate       0
Is Superhost             0
neighbourhood            0
Neighborhood Group       0
Country Code             0
Country                  0
Latitude                 0
Longitude                0
Is Exact Location        0
Property Type            0
Room Type                0
Accomodates              0
Bathrooms                0
Bedrooms                 0
Beds                     0
Price                    0
Guests Included          0
Min Nights               0
Reviews                  0
Overall Rating           0
Accuracy Rating          0
Cleanliness Rating       0
Checkin Rating           0
Communication Rating     0
Location Rating          0
Value Rating             0
Instant Bookable         0
Business Travel Ready    0
dtype: int64


In [10]:
high_cardinality_cols = [
    'neighbourhood'
]

min_frequency = 20

for col in high_cardinality_cols:

    counts = df_ml[col].value_counts()

    rare_categories = counts[counts < min_frequency].index

    df_ml[col] = df_ml[col].replace(rare_categories, 'Other')

In [11]:
categorical_cols = df_ml.select_dtypes(include=['object', 'str']).columns

df_final_ml = pd.get_dummies(
    df_ml,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print(df_final_ml.shape)

(4970, 79)


In [12]:
categorical_cols = df_ml.select_dtypes(include=['object', 'str']).columns

for col in categorical_cols:
    print(f"{col}: {df_ml[col].nunique()} unique values")

Host Response Time: 5 unique values
Is Superhost: 2 unique values
neighbourhood: 22 unique values
Neighborhood Group: 12 unique values
Country Code: 1 unique values
Country: 1 unique values
Is Exact Location: 2 unique values
Property Type: 21 unique values
Room Type: 3 unique values
Instant Bookable: 2 unique values
Business Travel Ready: 1 unique values


In [14]:
# Save dataset for regression models
regression_path = "../../data/processed/berlin_airbnb_regression.csv"

df_final_ml.to_csv(regression_path, index=False)

print(f"Regression dataset saved to:\n{regression_path}")
print(df_final_ml.shape)

Regression dataset saved to:
../../data/processed/berlin_airbnb_regression.csv
(4970, 79)


In [15]:
print(df_final_ml.columns.tolist())

['Host Response Rate', 'Latitude', 'Longitude', 'Accomodates', 'Bathrooms', 'Bedrooms', 'Beds', 'Price', 'Guests Included', 'Min Nights', 'Reviews', 'Overall Rating', 'Accuracy Rating', 'Cleanliness Rating', 'Checkin Rating', 'Communication Rating', 'Location Rating', 'Value Rating', 'Host Response Time_a few days or more', 'Host Response Time_within a day', 'Host Response Time_within a few hours', 'Host Response Time_within an hour', 'Is Superhost_t', 'neighbourhood_Alt-Treptow', 'neighbourhood_Charlottenburg', 'neighbourhood_Friedrichshain', 'neighbourhood_Kreuzberg', 'neighbourhood_Lichtenberg', 'neighbourhood_Mitte', 'neighbourhood_Moabit', 'neighbourhood_NeukÃ¶lln', 'neighbourhood_Other', 'neighbourhood_Pankow', 'neighbourhood_Prenzlauer Berg', 'neighbourhood_Reinickendorf', 'neighbourhood_Rummelsburg', 'neighbourhood_SchÃ¶neberg', 'neighbourhood_Steglitz', 'neighbourhood_Tempelhof', 'neighbourhood_Tiergarten', 'neighbourhood_Wedding', 'neighbourhood_WeiÃ\x9fensee', 'neighbourhood